# project_08_kras_binder — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [1]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

Python : 3.11.15
Platform: Linux-6.18.5-x86_64-with-glibc2.39
GPU    : NONE FOUND
Structure prediction on CPU is impractically slow.


## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [2]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

Note: you may need to restart the kernel to use updated packages.
Core install done.


In [3]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

# Environment stamp 2026-06-24T03:15:20 UTC
Bio            1.84
py3Dmol        2.4.0


numpy          2.4.6


pandas         3.0.3
matplotlib     3.11.0


seaborn        0.13.2
tqdm           4.68.3
requests       2.33.1


## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [4]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

Helpers ready: install_colabfold(), install_esmfold().


## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [5]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

seeds set to 0
logged: Ran 00_setup; environment stamped.


## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [6]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

Uncomment to mount Drive and set your working directory.


---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — KRAS biology, epitope/allele/nucleotide-state choice, binder metrics

**Standard slot:** *define & explore.* **For Project 08 this means:** understand why KRAS was
"undruggable", **choose the epitope (switch I/II or an allele pocket), the allele, and the
nucleotide state** on purpose, clean the KRAS G-domain, write down the binder metrics + cutoffs, and
run a deterministic **mock** mini-run (with a mock isoform panel) as your "hello-world" (D0).

Run `00_setup.ipynb` first in this session. A real binder campaign wants an **A100** (see
`MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab Pro / A100.

## Why KRAS is hard (and what we are designing)

KRAS is a small GTPase that cycles between a **GDP "off"** and a **GTP "on"** state; effectors (RAF,
PI3K) engage the "on" state. Activating codon-12 mutations (**G12C, G12D, G12V**) lock it on and drive
~25% of cancers. It was "undruggable" because the surface is small, smooth, charged, and pocket-poor.
The druggable handholds are the **switch I (~res 30–38)** and **switch II (~res 60–76)** regions, or an
**allele-specific** surface (e.g. the G12C cysteine). Two design decisions dominate everything:

1. **Nucleotide state** — the switch surface only exists in one state (GDP vs GTP/analog). Model the
   right one.
2. **Selectivity** — KRAS, HRAS, and NRAS are nearly identical across the switches, so a binder that
   hits KRAS usually hits all three. The centerpiece of this project is the **isoform-specificity
   panel** (notebook 04) and reasoning about **allele** selectivity.

## The binder metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the binder | thermostability / ΔG |
| **pae_interaction** | Å | AF2-Multimer error across the **binder–KRAS interface** (the key binder metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |
| TM-score | 0–1 | similarity to nearest known fold (<0.5 ≈ novel) | a pass/fail of correctness |
| **isoform selectivity gap** | Å | KRAS pae minus best off-target (HRAS/NRAS) pae | measured selectivity |

The shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10, rosetta_dG ≤ −30,
sc ≥ 0.6.** `pae_interaction` is the single most important binder metric — but a low value is
*confidence*, **not** affinity, and a positive selectivity gap is a *hypothesis* of selectivity. A
passing design is a **hypothesis** until SPR/BLI + the isoform panel.

## Setup paths

In [7]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_08_kras_binder/notebooks


## 1 · Target prep + epitope/allele/nucleotide-state choice

The design target is the **KRAS G-domain** in a chosen **nucleotide state**, and the hotspots are the
KRAS residues of your chosen epitope — **switch I (~30–38)**, **switch II (~60–76)**, or an
**allele pocket**. Fetch the candidate structures with `data/download_data.py` (4OBE WT / 6OIM G12C —
**verify on RCSB**, and add HRAS/NRAS for the panel), isolate the KRAS chain, keep the bound
nucleotide + Mg²⁺ where the switch depends on them, remove waters, and read the hotspots off **that**
structure/state.

Below we just *declare* an EXAMPLE choice so the notebook runs end-to-end; **replace it with the
residues, allele, and state you derive** (numbering depends on the PDB you verify).

In [8]:
import binder_tools as bt

TARGET = "KRAS"                 # cleaned KRAS G-domain (you produce this from 4OBE / 6OIM)
ALLELE = "G12C"                 # EXAMPLE allele you designed against — e.g. "WT", "G12C", "G12D"
NUCLEOTIDE_STATE = "GDP"        # EXAMPLE state — "GDP" ("off") or "GTP"/"GppNHp" ("on"); the switch surface depends on it
# EXAMPLE switch I/II hotspots — VERIFY/REPLACE from the actual KRAS structure + state (data/README.md).
# These are placeholders so the plumbing runs; real numbering depends on the PDB chain you clean.
HOTSPOTS = bt.parse_hotspots("A32,A35,A38,A60,A71")   # EXAMPLE_DATA: switch I (~30-38) + switch II (~60-76)
print("target          :", TARGET)
print("allele          :", ALLELE, " (EXAMPLE — record the allele you actually designed against)")
print("nucleotide state:", NUCLEOTIDE_STATE, " (EXAMPLE — the switch surface only exists in one state)")
print("hotspots        :", HOTSPOTS, " (EXAMPLE — replace with your verified switch I/II residues)")

target          : KRAS
allele          : G12C  (EXAMPLE — record the allele you actually designed against)
nucleotide state: GDP  (EXAMPLE — the switch surface only exists in one state)
hotspots        : ('A32', 'A35', 'A38', 'A60', 'A71')  (EXAMPLE — replace with your verified switch I/II residues)


## 2 · Mock hello-world: a tiny two-paradigm mini-run

`scripts/binder_tools.py` exposes both paradigms behind one API:
`generate_binders_bindcraft(...)` and `generate_binders_rfdiffusion(...)`, plus `af2_multimer(...)`
(the scorer) and `isoform_specificity(...)` / `specificity_panel(...)` (the KRAS-specific selectivity
helpers). The **mock** backend is deterministic and GPU-free so you can develop the plumbing.
**Never report mock numbers as real** — they are `SYNTHETIC` by construction, and there is no K_D anywhere.

In [9]:
# A few designs from each paradigm, scored by mock AF2-Multimer. All numbers are SYNTHETIC.
bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=3, tool="mock", allele=ALLELE, nucleotide_state=NUCLEOTIDE_STATE)
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=3, tool="mock", allele=ALLELE, nucleotide_state=NUCLEOTIDE_STATE)
bt.score_designs(bc, tool="mock")
bt.score_designs(rf, tool="mock")

d = bc[0]
print("example BindCraft design:")
print("  id    :", d.design_id)
print("  allele:", d.allele, " state:", d.nucleotide_state)
print("  len   :", d.length, "aa")
print("  seq   :", d.sequence)
print("  pae_interaction =", d.pae_interaction, " scrmsd =", d.scrmsd,
      " sc =", d.shape_complementarity, " (SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.")

example BindCraft design:
  id    : EXAMPLE_DATA_bindcraft_0000
  allele: G12C  state: GDP
  len   : 58 aa
  seq   : AGHIFVDEFVDNALREFVDSTVMIFGHSTVREKLWIFGDSPVMNAGWNPGHYPQHIFQ
  pae_interaction = 8.0  scrmsd = 3.84  sc = 0.49  (SYNTHETIC)
  synthetic flag  : True -> SYNTHETIC — mock backend, not a real design/prediction

Reminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.


## 3 · Mock isoform-specificity panel (the KRAS-specific check)

The point of this project is not just "binds KRAS" but "binds KRAS and **not** HRAS/NRAS".
`specificity_panel()` re-scores a binder against each RAS isoform and computes the **selectivity gap**
(KRAS pae minus the best off-target pae; positive ⇒ some selectivity). Because the isoforms are nearly
identical across the switches, real selectivity is hard — here it is a deterministic SYNTHETIC proxy
you will replace with real AF2-Multimer runs vs HRAS/NRAS on Colab.

In [10]:
for b in bc[:3]:
    panel = bt.specificity_panel(b, tool="mock")
    print(f"{b.design_id}: per-isoform pae = {panel['per_isoform']}  "
          f"selectivity_gap = {panel['selectivity_gap']}  selective = {panel['selective']}  (SYNTHETIC)")
print("\nDelta > 0 ⇒ KRAS scores better than the best off-target. On Colab, replace mock with real "
      "AF2-Multimer vs verified HRAS/NRAS structures. A pan-RAS binder (gap ~ 0) is a weaker result — report it.")

EXAMPLE_DATA_bindcraft_0000: per-isoform pae = {'KRAS': 8.0, 'HRAS': 17.0, 'NRAS': 17.0}  selectivity_gap = 9.0  selective = True  (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0001: per-isoform pae = {'KRAS': 5.0, 'HRAS': 15.0, 'NRAS': 5.0}  selectivity_gap = 0.0  selective = False  (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0002: per-isoform pae = {'KRAS': 6.0, 'HRAS': 14.0, 'NRAS': 11.0}  selectivity_gap = 5.0  selective = True  (SYNTHETIC)

Delta > 0 ⇒ KRAS scores better than the best off-target. On Colab, replace mock with real AF2-Multimer vs verified HRAS/NRAS structures. A pan-RAS binder (gap ~ 0) is a weaker result — report it.


## 4 · Epitope coverage / effector-competition proxy

A switch-region binder could **block RAF/effector engagement** if it covers enough of the switch
footprint. `hotspot_overlap()` is a geometry proxy (fraction of chosen-epitope hotspots contacted) — a
teaching stand-in for the effector-competition assay in notebook 04. Higher ⇒ more likely to block
(not a guarantee).

In [11]:
for b in bc[:3]:
    ov = bt.hotspot_overlap(b.contact_residues, HOTSPOTS)
    print(f"{b.design_id}: contacts {b.contact_residues} -> switch-footprint coverage = {ov} (SYNTHETIC)")

EXAMPLE_DATA_bindcraft_0000: contacts ('A32', 'A35', 'A38') -> switch-footprint coverage = 0.6 (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0001: contacts ('A32', 'A35', 'A38') -> switch-footprint coverage = 0.6 (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0002: contacts ('A32', 'A35', 'A38') -> switch-footprint coverage = 0.6 (SYNTHETIC)


## Visualize a binder–KRAS complex (py3Dmol)

Use this to eyeball a predicted binder–KRAS complex once you have a real PDB (from AF2-Multimer) — and
to overlay KRAS/HRAS/NRAS at the switch regions when reasoning about selectivity.

In [12]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer prediction writes a complex PDB):
# show_complex("results/af2/top_complex.pdb")
print("show_complex(pdb_path) ready.")

show_complex(pdb_path) ready.


## D0 checklist
- [ ] KRAS accessions verified on RCSB (4OBE/6OIM are candidates); **HRAS/NRAS** added for the panel; KRAS chain identified.
- [ ] **Epitope, allele, and nucleotide state chosen and justified**; cleaned KRAS target + hotspot list (derived from that structure/state, not invented).
- [ ] One-paragraph definition of each binder metric **with** its "does not mean" note (incl. the selectivity gap).
- [ ] Reproduced mock mini-run (both paradigms) + mock isoform panel, with metrics printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria (incl. a selectivity target) + controls; `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the two-paradigm binder campaign at the chosen KRAS surface.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — two-paradigm binder design vs KRAS

**Standard slot:** *design campaign.* **For Project 08 this is the core:** run **both** paradigms
against your chosen KRAS surface (allele + nucleotide state) and assemble their pools (D2):
- **BindCraft** (one-shot hallucination, AF2-Multimer in the loop) — **50–200** designs.
- **RFdiffusion binder mode → ProteinMPNN** — **500–1000** backbones → sequences.

Then score every design with **AF2-Multimer** (`pae_interaction` is the key binder metric).

> **Compute honesty:** a real campaign at this scale wants an **A100** (Colab Pro+ or a cluster).
> Free **T4** = a *small fallback* (FreeBindCraft, small `num_designs`, a small RFdiffusion batch +
> ESMFold triage). The cells below run on the deterministic **mock** backend so the plumbing executes
> anywhere; the real calls + A100 notes are shown alongside. Run `00_setup.ipynb` first.

## Setup paths

In [13]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_08_kras_binder/notebooks


## Version-verify the pinned upstreams (tools change!)

The binder tools live in fast-moving upstream repos. **Pin commits** and **verify the URLs still
exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin and log
it). The generation itself needs an A100; this check needs nothing.

In [14]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   BindCraft      https://github.com/martinpacesa/BindCraft        # e.g. pin <commit>
#   FreeBindCraft  https://github.com/cytokineking/FreeBindCraft     # free-tier fallback — VERIFY it exists; pin <commit>
#   RFdiffusion    https://github.com/RosettaCommons/RFdiffusion     # pin <commit>
#   ColabDesign    https://github.com/sokrypton/ColabDesign          # RFdiffusion-binder + ProteinMPNN; pin <commit>
#   ColabFold      https://github.com/sokrypton/ColabFold            # AF2-Multimer (+ isoform panel); pin <commit>
PINNED = {
    "BindCraft":     "https://github.com/martinpacesa/BindCraft",
    "FreeBindCraft": "https://github.com/cytokineking/FreeBindCraft",
    "RFdiffusion":   "https://github.com/RosettaCommons/RFdiffusion",
    "ColabDesign":   "https://github.com/sokrypton/ColabDesign",
    "ColabFold":     "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:14s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:14s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.")

  [200] BindCraft      https://github.com/martinpacesa/BindCraft


  [200] FreeBindCraft  https://github.com/cytokineking/FreeBindCraft


  [200] RFdiffusion    https://github.com/RosettaCommons/RFdiffusion


  [200] ColabDesign    https://github.com/sokrypton/ColabDesign


  [200] ColabFold      https://github.com/sokrypton/ColabFold

Non-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.
FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.


## 1 · Define the campaign

Same target + epitope + allele + nucleotide state as notebook 01. Set honest campaign sizes; the cells
run on `mock` so they execute anywhere. On Colab (A100) switch `TOOL_*` to the real backends — and
**shrink the numbers on a T4** (FreeBindCraft, a small RFdiffusion batch).

In [15]:
import binder_tools as bt
import pandas as pd

TARGET = "KRAS"
ALLELE = "G12C"                 # EXAMPLE — the allele you designed against
NUCLEOTIDE_STATE = "GDP"        # EXAMPLE — the modeled state
HOTSPOTS = bt.parse_hotspots("A32,A35,A38,A60,A71")   # EXAMPLE — replace with your verified switch I/II residues

# Honest campaign sizes (catalog): BindCraft 50-200, RFdiffusion 500-1000 backbones.
# We use small mock counts here so the dry run is fast; scale up with the real backend on A100.
N_BINDCRAFT   = 60      # -> 50-200 on A100; fewer (FreeBindCraft) on T4
N_RFDIFFUSION = 200     # -> 500-1000 backbones on A100; small batch on T4

TOOL_BINDCRAFT   = "mock"   # -> "bindcraft" / "freebindcraft" on Colab
TOOL_RFDIFFUSION = "mock"   # -> "rfdiffusion" on Colab
TOOL_AF2         = "mock"   # -> "af2" (ColabFold AF2-Multimer) on Colab

print(f"BindCraft   : n={N_BINDCRAFT}  tool={TOOL_BINDCRAFT}")
print(f"RFdiffusion : n={N_RFDIFFUSION} tool={TOOL_RFDIFFUSION}")
print(f"AF2-Multimer: tool={TOOL_AF2}")
print("target/allele/state:", TARGET, ALLELE, NUCLEOTIDE_STATE)
print("hotspots    :", HOTSPOTS)

BindCraft   : n=60  tool=mock
RFdiffusion : n=200 tool=mock
AF2-Multimer: tool=mock
target/allele/state: KRAS G12C GDP
hotspots    : ('A32', 'A35', 'A38', 'A60', 'A71')


## 2 · Paradigm #1 — BindCraft campaign

One-shot hallucination with AF2-Multimer in the loop. On A100 this produces 50–200 binders pre-filtered
on interface confidence; we still re-score with AF2-Multimer so the head-to-head with RFdiffusion is
apples-to-apples. The `mock` backend returns deterministic `SYNTHETIC` designs.

In [16]:
# Real call (Colab, A100): bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT,
#   tool="bindcraft", allele=ALLELE, nucleotide_state=NUCLEOTIDE_STATE)
#   free-tier fallback: tool="freebindcraft", smaller N. See MANUAL.md §2 / scripts/binder_tools.py TODOs.
bindcraft = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool=TOOL_BINDCRAFT,
                                          allele=ALLELE, nucleotide_state=NUCLEOTIDE_STATE)
bt.score_designs(bindcraft, tool=TOOL_AF2)     # AF2-Multimer -> pae_interaction, plddt, scrmsd, sc
print(f"BindCraft pool: {len(bindcraft)} designs (tool={TOOL_BINDCRAFT}; SYNTHETIC if mock)")
print("example:", bindcraft[0].design_id, "pae_interaction=", bindcraft[0].pae_interaction)

BindCraft pool: 60 designs (tool=mock; SYNTHETIC if mock)
example: EXAMPLE_DATA_bindcraft_0000 pae_interaction= 8.0


## 3 · Paradigm #2 — RFdiffusion binder campaign → ProteinMPNN

Diffuse binder backbones docked at the switch I/II hotspots, then ProteinMPNN designs sequences, then
AF2-Multimer re-predicts each complex. On A100 this is 500–1000 backbones (the per-backbone hit rate is
low on a small/smooth surface like KRAS — that is normal). The `mock` backend stands in for the whole chain.

In [17]:
# Real call (Colab, A100): bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION,
#   tool="rfdiffusion", mpnn_temperature=0.1, num_seq_per_backbone=8, allele=ALLELE,
#   nucleotide_state=NUCLEOTIDE_STATE). AF2-Multimer is the slow step.
rfdiff = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION, tool=TOOL_RFDIFFUSION,
                                         allele=ALLELE, nucleotide_state=NUCLEOTIDE_STATE)
bt.score_designs(rfdiff, tool=TOOL_AF2)
print(f"RFdiffusion pool: {len(rfdiff)} designs (tool={TOOL_RFDIFFUSION}; SYNTHETIC if mock)")
print("example:", rfdiff[0].design_id, "pae_interaction=", rfdiff[0].pae_interaction)

RFdiffusion pool: 200 designs (tool=mock; SYNTHETIC if mock)
example: EXAMPLE_DATA_rfdiffusion_0000 pae_interaction= 4.0


## 4 · Assemble + persist both pools

Write one tidy CSV per paradigm (plus a combined one). These feed notebook 03 (the shared filter). We
add an EXAMPLE physics column (`rosetta_dG`) here so the binder physics layer has something to act on in
the dry run — on Colab these come from FreeBindCraft/PyRosetta; for `mock` they are SYNTHETIC. We also
carry `allele` + `nucleotide_state` on every row so the campaign is fully described.

In [18]:
import pandas as pd

def pool_to_df(designs):
    rows = []
    for d in designs:
        # In the mock dry run we attach an EXAMPLE_DATA interface energy so Layer 3 (physics) is
        # exercised. On Colab, replace with the real FreeBindCraft/PyRosetta rosetta_dG + solubility.
        rdg = -45.0 + (bt._hashints("dG", d.design_id) % 40)   # SYNTHETIC, range ~ -45..-6 REU
        rows.append(dict(
            design_id=d.design_id, paradigm=d.paradigm, target=d.target,
            allele=d.allele, nucleotide_state=d.nucleotide_state,
            length=d.length, sequence=d.sequence,
            plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
            shape_complementarity=d.shape_complementarity,
            rosetta_dG=round(float(rdg), 2), solubility=0.3,
            contact_residues=",".join(d.contact_residues),
            hotspot_overlap=bt.hotspot_overlap(d.contact_residues, d.hotspots),
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

df_bc = pool_to_df(bindcraft); df_bc.to_csv("results/bindcraft_designs.csv", index=False)
df_rf = pool_to_df(rfdiff);    df_rf.to_csv("results/rfdiffusion_designs.csv", index=False)
combined = pd.concat([df_bc, df_rf], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

print("wrote results/bindcraft_designs.csv   ", df_bc.shape)
print("wrote results/rfdiffusion_designs.csv ", df_rf.shape)
print("wrote results/all_designs.csv         ", combined.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.")
combined.head(4)

wrote results/bindcraft_designs.csv    (60, 16)
wrote results/rfdiffusion_designs.csv  (200, 16)
wrote results/all_designs.csv          (260, 16)

ALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.


,design_id,paradigm,target,allele,nucleotide_state,length,sequence,plddt,pae_interaction,scrmsd,shape_complementarity,rosetta_dG,solubility,contact_residues,hotspot_overlap,synthetic
0,EXAMPLE_DATA_bindcraft_0000,bindcraft,KRAS,G12C,GDP,58,AGHIFVDEFVDNALREFVDSTVMIFGHSTVREKLWIFGDSPVMNAG...,74.0,8.0,3.84,0.49,-38.0,0.3,"A32,A35,A38",0.6,True
1,EXAMPLE_DATA_bindcraft_0001,bindcraft,KRAS,G12C,GDP,52,FGDNFQDNTGWSKGHYTLHYALDIACDNFGHEFCDEPQHIKCRNFG...,83.0,5.0,4.13,0.58,-19.0,0.3,"A32,A35,A38",0.6,True
2,EXAMPLE_DATA_bindcraft_0002,bindcraft,KRAS,G12C,GDP,66,KCHYALMNKQDIPLMYFLMSKVDIPLMYFCRSPLMYPQMSTGMETV...,78.0,6.0,0.88,0.68,-35.0,0.3,"A32,A35,A38",0.6,True
3,EXAMPLE_DATA_bindcraft_0003,bindcraft,KRAS,G12C,GDP,77,VMITCMYKVRYPQRNACMSFGHSKGMEPCWIKGMNFLRNTVRIFLH...,82.0,10.0,4.22,0.87,-38.0,0.3,"A32,A35,A38,A60",0.8,True


## D2 checklist
- [ ] BindCraft pool generated at honest scale (50–200 on A100; FreeBindCraft/small on T4).
- [ ] RFdiffusion-binder pool generated (500–1000 backbones → ProteinMPNN on A100).
- [ ] Every design scored by AF2-Multimer (`pae_interaction` parsed); both pools written to `results/`.
- [ ] Design log: every config + seed + **allele + nucleotide state** + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured; 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on both pools.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25 projects
use. **For Project 08** you build `fp.Design` **binder** objects from both pools, call
`fp.run_pipeline(..., design_type="binder")`, and `fp.report(...)` the survival funnel + ranked CSV,
**per paradigm** so the head-to-head is fair (D3 part 1). The KRAS-specific **isoform-specificity**
analysis comes next, in notebook 04.

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` exist.

## Setup paths

In [19]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_08_kras_binder/notebooks


## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"binder"` cutoffs: scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6.

In [20]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

Loaded shared filtering_pipeline from: /home/user/biofx_python/denovo_protein_design_course/shared/filtering_pipeline.py
binder cutoffs: {'scrmsd': 2.5, 'plddt': 80, 'pae': 10, 'rosetta_dG': -30, 'sc': 0.6}


## Build `Design` (binder) objects from the pools

Map each pool row onto `fp.Design` with `design_type="binder"`. The binder metrics drive the layers:
`scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency), and `rosetta_dG`/`shape_complementarity`/`solubility`
(Layer 3 physics). We keep `paradigm`, `allele`, `nucleotide_state`, and `hotspot_overlap` in `extra` for
the head-to-head + isoform + effector analysis in notebook 04. (Mock has no independent orthogonal
predictor, so we run Layers 1+3 here; on Colab add a second predictor for Layer 2.)

In [21]:
import os
import pandas as pd

# Regenerate the pools if a fresh session lost them (deterministic mock).
if not (os.path.exists("results/bindcraft_designs.csv") and os.path.exists("results/rfdiffusion_designs.csv")):
    import binder_tools as bt
    TARGET, HOTSPOTS = "KRAS", bt.parse_hotspots("A32,A35,A38,A60,A71")
    ALLELE, STATE = "G12C", "GDP"
    bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=60, tool="mock", allele=ALLELE, nucleotide_state=STATE);  bt.score_designs(bc, tool="mock")
    rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=200, tool="mock", allele=ALLELE, nucleotide_state=STATE); bt.score_designs(rf, tool="mock")
    def _q(designs, p):
        rows=[dict(design_id=d.design_id, paradigm=d.paradigm, allele=d.allele, nucleotide_state=d.nucleotide_state,
                   length=d.length, sequence=d.sequence,
                   plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
                   shape_complementarity=d.shape_complementarity,
                   rosetta_dG=round(-45.0+(bt._hashints("dG",d.design_id)%40),2), solubility=0.3,
                   hotspot_overlap=bt.hotspot_overlap(d.contact_residues,d.hotspots), synthetic=d.synthetic)
              for d in designs]
        pd.DataFrame(rows).to_csv(p, index=False)
    _q(bc, "results/bindcraft_designs.csv"); _q(rf, "results/rfdiffusion_designs.csv")

df_bc = pd.read_csv("results/bindcraft_designs.csv")
df_rf = pd.read_csv("results/rfdiffusion_designs.csv")

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3),
        extra={"paradigm": r.get("paradigm"), "allele": r.get("allele"),
               "nucleotide_state": r.get("nucleotide_state"), "hotspot_overlap": r.get("hotspot_overlap")},
    )

binders_bc = [row_to_binder(r) for _, r in df_bc.iterrows()]
binders_rf = [row_to_binder(r) for _, r in df_rf.iterrows()]
print(f"built {len(binders_bc)} BindCraft + {len(binders_rf)} RFdiffusion binder Designs")

built 60 BindCraft + 200 RFdiffusion binder Designs


## Run the pipeline — per paradigm (fair head-to-head)

`run_pipeline(design_type="binder")` applies the binder cutoffs in order and returns a ranked DataFrame
with survival counts in `df.attrs`. We run **each paradigm separately** so the survival-at-each-layer
funnels are comparable. We use Layers 1+3 here (mock has no independent orthogonal source; add Layer 2
on Colab with a second predictor).

In [22]:
def run_one(designs, label):
    df = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
    df["paradigm"] = label
    surv = df.attrs["survival"]; n = df.attrs["n_total"]
    passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: {n} designs, survival {surv}, all-layers hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")
    return df

ranked_bc = run_one(binders_bc, "bindcraft")
ranked_rf = run_one(binders_rf, "rfdiffusion")

ranked = pd.concat([ranked_bc, ranked_rf], ignore_index=True).sort_values(
    ["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/all_ranked.csv", index=False)
print("\nwrote results/all_ranked.csv", ranked.shape)
ranked.head(10)[["design_id", "paradigm", "layers_passed", "score",
                 "scrmsd", "plddt", "pae_interaction", "rosetta_dG"]]

bindcraft   : 60 designs, survival {'L1': 12, 'L3': 1}, all-layers hit rate = 1/60 (1.7%)
rfdiffusion : 200 designs, survival {'L1': 25, 'L3': 7}, all-layers hit rate = 7/200 (3.5%)

wrote results/all_ranked.csv (260, 21)


,design_id,paradigm,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG
0,EXAMPLE_DATA_rfdiffusion_0079,rfdiffusion,3,4.5133,1.05,85.0,5.0,-44.0
1,EXAMPLE_DATA_rfdiffusion_0167,rfdiffusion,3,3.8033,1.34,84.0,8.0,-36.0
2,EXAMPLE_DATA_rfdiffusion_0007,rfdiffusion,3,3.7667,1.65,85.0,7.0,-45.0
3,EXAMPLE_DATA_rfdiffusion_0161,rfdiffusion,3,3.7000,1.41,91.0,9.0,-35.0
4,EXAMPLE_DATA_bindcraft_0028,bindcraft,3,3.2133,2.17,87.0,5.0,-34.0
5,EXAMPLE_DATA_rfdiffusion_0042,rfdiffusion,3,3.1700,2.12,82.0,6.0,-36.0
6,EXAMPLE_DATA_rfdiffusion_0051,rfdiffusion,3,3.1167,1.82,92.0,10.0,-30.0
7,EXAMPLE_DATA_rfdiffusion_0131,rfdiffusion,3,2.6967,2.38,88.0,10.0,-39.0
8,EXAMPLE_DATA_rfdiffusion_0034,rfdiffusion,1,3.9700,0.80,90.0,6.0,-31.0
9,EXAMPLE_DATA_rfdiffusion_0178,rfdiffusion,1,3.9533,0.81,81.0,5.0,-31.0


## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the **combined**
pool for one comparable figure; the per-paradigm runs above are the rigorous version. Read the bars as a
funnel: steep drops show which layer discriminates.

In [23]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

all_binders = binders_bc + binders_rf
df_all = fp.run_pipeline(all_binders, design_type="binder", use_layers=(1, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p08")
print("\nsaved results/p08_survival.png + results/p08_ranked.csv")
top

Total designs: 260
  L1 survivors: 37  (14.2%)
  L3 survivors: 8  (3.1%)



saved results/p08_survival.png + results/p08_ranked.csv


,design_id,design_type,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG,tm_to_pdb
0,EXAMPLE_DATA_rfdiffusion_0079,binder,3,4.5133,1.05,85.0,5.0,-44.0,None
1,EXAMPLE_DATA_rfdiffusion_0167,binder,3,3.8033,1.34,84.0,8.0,-36.0,None
2,EXAMPLE_DATA_rfdiffusion_0007,binder,3,3.7667,1.65,85.0,7.0,-45.0,None
3,EXAMPLE_DATA_rfdiffusion_0161,binder,3,3.7000,1.41,91.0,9.0,-35.0,None
4,EXAMPLE_DATA_bindcraft_0028,binder,3,3.2133,2.17,87.0,5.0,-34.0,None
5,EXAMPLE_DATA_rfdiffusion_0042,binder,3,3.1700,2.12,82.0,6.0,-36.0,None
6,EXAMPLE_DATA_rfdiffusion_0051,binder,3,3.1167,1.82,92.0,10.0,-30.0,None
7,EXAMPLE_DATA_rfdiffusion_0131,binder,3,2.6967,2.38,88.0,10.0,-39.0,None
8,EXAMPLE_DATA_rfdiffusion_0034,binder,1,3.9700,0.80,90.0,6.0,-31.0,None
9,EXAMPLE_DATA_rfdiffusion_0178,binder,1,3.9533,0.81,81.0,5.0,-31.0,None


## Honest hit-rate accounting (per paradigm)

Report `N passing all layers / N generated` for **each** paradigm — this is the number the head-to-head
and the selectivity analysis in notebook 04 build on. Remember: survival is *enrichment*, not
*correctness*, and it says nothing about selectivity. Mock numbers are SYNTHETIC.

In [24]:
for label, df in [("bindcraft", ranked_bc), ("rfdiffusion", ranked_rf)]:
    n = len(df); passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: layers_passed distribution {df['layers_passed'].value_counts().sort_index().to_dict()}")
    print(f"{'':12s}  all-layers survivors = {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")

bindcraft   : layers_passed distribution {0: 48, 1: 11, 3: 1}
              all-layers survivors = 1/60 (1.7%)  [SYNTHETIC if mock]
rfdiffusion : layers_passed distribution {0: 175, 1: 18, 3: 7}
              all-layers survivors = 7/200 (3.5%)  [SYNTHETIC if mock]


## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Survival-at-each-layer reported **per paradigm** (funnel figure `results/p08_survival.png`).
- [ ] Honest hit-rate accounting (N pass / N generated) for BindCraft and RFdiffusion.
- [ ] Mapping assumptions (which fields → which `Design` attributes) written down; allele/state carried through.

**Next:** `04_validate.ipynb` — the BindCraft-vs-RFdiffusion benchmark + the **isoform-specificity** analysis.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — BindCraft vs RFdiffusion + isoform specificity + effector competition

**Standard slot:** *validate (in silico).* **For Project 08 this is the core analysis:** the
head-to-head between the two paradigms (hit rate, interface energy, novelty), **the
isoform-specificity panel (KRAS vs HRAS/NRAS, the selectivity gap)** — the scientific heart of this
project — and **effector-competition reasoning (block RAF)** `[extension]`, with publication-style
figures (D3 part 2).

Needs `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` + `results/all_ranked.csv`
(from notebooks 02–03).

## Setup paths

In [25]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_08_kras_binder/notebooks


## 1 · Head-to-head hit rate + interface energy

Compare the two paradigms on (a) all-layers **hit rate** and (b) the **interface-energy** (`rosetta_dG`)
distribution of survivors. A fair comparison filters both identically (notebook 03) and reports the
*distribution*, not the single best. Mock numbers are SYNTHETIC.

In [26]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
print("paradigms:", ranked["paradigm"].value_counts().to_dict())

summary = []
for p, g in ranked.groupby("paradigm"):
    n = len(g); passed = int((g["layers_passed"] >= 3).sum())
    summary.append(dict(paradigm=p, n=n, all_layers_survivors=passed,
                        hit_rate_pct=round(100*passed/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_dG=round(float(g["rosetta_dG"].median()), 2)))
summary = pd.DataFrame(summary)
print("\nhead-to-head summary (SYNTHETIC if mock):")
print(summary.to_string(index=False))

paradigms: {'rfdiffusion': 200, 'bindcraft': 60}

head-to-head summary (SYNTHETIC if mock):
   paradigm   n  all_layers_survivors  hit_rate_pct  median_pae  median_dG
  bindcraft  60                     1           1.7        10.0      -31.0
rfdiffusion 200                     7           3.5        12.0      -26.5


In [27]:
# Interface-energy distribution per paradigm (survivors).
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for p, g in ranked.groupby("paradigm"):
    surv = g[g["layers_passed"] >= 3]
    ax[0].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=p)
    ax[1].hist(surv["rosetta_dG"].dropna(), bins=15, alpha=0.5, label=p)
ax[0].set_xlabel("pae_interaction (Å, lower better)"); ax[0].set_ylabel("designs"); ax[0].set_title("AF2-Multimer pae_interaction"); ax[0].legend()
ax[1].set_xlabel("rosetta_dG (REU, more negative better)"); ax[1].set_title("Interface energy (survivors)"); ax[1].legend()
fig.suptitle("BindCraft vs RFdiffusion vs KRAS (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p08_headtohead.png", dpi=150); plt.show()
print("saved results/p08_headtohead.png")

saved results/p08_headtohead.png


## 2 · Isoform-specificity panel — KRAS vs HRAS/NRAS (the centerpiece)

The real question for KRAS is **selectivity**: a binder that hits KRAS *and* HRAS/NRAS is far less
useful (and more toxic) than a selective one. For each survivor we re-score the binder against each RAS
isoform with the **same** AF2-Multimer scorer (`specificity_panel`, mock here) and compute the
**selectivity gap** = KRAS pae − best off-target pae (positive ⇒ KRAS scores better ⇒ some selectivity).
Because the isoforms are nearly identical across the switches, **expect this to be hard** — report the
gap distribution honestly, including binders that turn out pan-RAS. On Colab, replace `tool="mock"`
with `tool="af2"` and verified HRAS/NRAS structures.

In [28]:
import binder_tools as bt

# Rebuild the survivor sequences from the pools (we need the sequence to re-score vs each isoform).
bc = pd.read_csv("results/bindcraft_designs.csv")
rf = pd.read_csv("results/rfdiffusion_designs.csv")
pools = pd.concat([bc, rf], ignore_index=True).set_index("design_id")

surv = ranked[ranked["layers_passed"] >= 3].copy()
rows = []
for _, r in surv.iterrows():
    did = r["design_id"]
    if did not in pools.index:
        continue
    seq = str(pools.loc[did, "sequence"])
    # Reconstruct a lightweight BinderDesign so the helper has allele/state/hotspots context.
    d = bt.BinderDesign(design_id=did, sequence=seq, paradigm=r["paradigm"], target="KRAS",
                        allele=str(pools.loc[did].get("allele", "WT")),
                        nucleotide_state=str(pools.loc[did].get("nucleotide_state", "GDP")),
                        hotspots=bt.parse_hotspots(str(pools.loc[did].get("contact_residues", "") or "")))
    panel = bt.specificity_panel(d, tool="mock")   # -> tool="af2" on Colab with verified HRAS/NRAS
    rows.append(dict(design_id=did, paradigm=r["paradigm"],
                     kras_pae=panel["kras_pae"], best_offtarget_pae=panel["best_offtarget_pae"],
                     selectivity_gap=panel["selectivity_gap"], selective=panel["selective"]))
spec = pd.DataFrame(rows)
spec.to_csv("results/isoform_specificity.csv", index=False)
print("wrote results/isoform_specificity.csv", spec.shape, "(SYNTHETIC if mock)")
if len(spec):
    for p, g in spec.groupby("paradigm"):
        print(f"  {p:12s}: median selectivity_gap = {g['selectivity_gap'].median():.2f}  "
              f"selective fraction = {g['selective'].mean():.2f}  (n={len(g)})")
    print("\nA gap ~ 0 ⇒ pan-RAS (binds HRAS/NRAS too) — a weaker result. Report the full distribution.")

wrote results/isoform_specificity.csv (8, 6) (SYNTHETIC if mock)
  bindcraft   : median selectivity_gap = -2.00  selective fraction = 0.00  (n=1)
  rfdiffusion : median selectivity_gap = -2.00  selective fraction = 0.14  (n=7)

A gap ~ 0 ⇒ pan-RAS (binds HRAS/NRAS too) — a weaker result. Report the full distribution.


In [29]:
# Selectivity-gap distribution per paradigm.
if len(spec):
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for p, g in spec.groupby("paradigm"):
        ax.hist(g["selectivity_gap"].dropna(), bins=15, alpha=0.5, label=p)
    ax.axvline(0, color="k", ls="--", lw=1, label="pan-RAS (gap=0)")
    ax.set_xlabel("selectivity gap = KRAS pae − best off-target pae (Å; >0 = selective)")
    ax.set_ylabel("survivors"); ax.set_title("Isoform selectivity (EXAMPLE_DATA if mock)"); ax.legend()
    plt.tight_layout(); plt.savefig("results/p08_selectivity.png", dpi=150); plt.show()
    print("saved results/p08_selectivity.png")
else:
    print("No survivors to plot — loosen the dry-run sizes or check notebook 03.")

saved results/p08_selectivity.png


## 3 · Novelty `[extension]`

Novelty = TM-score of each binder to its nearest natural fold (Foldseek/TM-align; `< 0.5` ≈ novel). On
Colab, compute it per design and compare the two paradigms' novelty distributions. Here we scaffold the
analysis (mock has no real structures), so we just show where it plugs in.

In [30]:
# Scaffold: on Colab, run Foldseek/TM-align on each predicted binder backbone -> tm_to_pdb,
# then compare distributions across paradigms (novel == tm_to_pdb < 0.5).
if "tm_to_pdb" in ranked.columns and ranked["tm_to_pdb"].notna().any():
    for p, g in ranked.groupby("paradigm"):
        novel = (g["tm_to_pdb"] < 0.5).mean()
        print(f"{p:12s}: novel fraction (TM<0.5) = {novel:.2f}")
else:
    print("Novelty scaffold — populate tm_to_pdb with Foldseek/TM-align on Colab, then compare paradigms.")

Novelty scaffold — populate tm_to_pdb with Foldseek/TM-align on Colab, then compare paradigms.


## 4 · Effector-competition vs RAF `[extension]`

A switch-region binder could **block RAF/effector engagement** if it covers enough of the switch
footprint. `hotspot_overlap` (notebook 02) is our geometry proxy: the fraction of switch I/II hotspots
the binder contacts. Higher ⇒ more likely to occlude the effector interface. Compare the survivors'
coverage across paradigms — a strong interface that *misses* the switch regions won't block RAF. (On
Colab, model the binder + RAF-RBD competition directly for a stronger test.)

In [31]:
bc = pd.read_csv("results/bindcraft_designs.csv")
rf = pd.read_csv("results/rfdiffusion_designs.csv")
pools2 = pd.concat([bc, rf], ignore_index=True)

ov = pools2.set_index("design_id")["hotspot_overlap"]
ranked["hotspot_overlap"] = ranked["design_id"].map(ov)
surv2 = ranked[ranked["layers_passed"] >= 3]

print("switch-footprint coverage of all-layers survivors (SYNTHETIC if mock):")
for p, g in surv2.groupby("paradigm"):
    print(f"  {p:12s}: median switch-footprint coverage = {g['hotspot_overlap'].median():.2f}  (n={len(g)})")

# "Likely effector blockers" = survivors that also cover enough of the switch footprint.
BLOCK_OVERLAP = 0.5
blockers = surv2[surv2["hotspot_overlap"] >= BLOCK_OVERLAP]
print(f"\nlikely effector blockers (survivor AND switch coverage>={BLOCK_OVERLAP}): {len(blockers)}")
print(blockers.groupby("paradigm").size().to_dict())

switch-footprint coverage of all-layers survivors (SYNTHETIC if mock):
  bindcraft   : median switch-footprint coverage = 0.20  (n=1)
  rfdiffusion : median switch-footprint coverage = 0.60  (n=7)

likely effector blockers (survivor AND switch coverage>=0.5): 4
{'rfdiffusion': 4}


## 5 · Select the top 10–20 per paradigm (prefer selective blockers)

The D★ deliverable wants the **top 10–20 each**. Rank survivors by the composite score and, as
tie-breakers, prefer a higher **selectivity gap** (KRAS-selective) and higher switch-footprint coverage
(effector blocker). Save the shortlist for the validation plan (notebook 05).

In [32]:
# Join the selectivity gap onto the ranked survivors.
if len(spec):
    gap = spec.set_index("design_id")["selectivity_gap"]
    ranked["selectivity_gap"] = ranked["design_id"].map(gap)
else:
    ranked["selectivity_gap"] = np.nan

top_per = []
for p, g in ranked.groupby("paradigm"):
    g2 = g[g["layers_passed"] >= 3].sort_values(
        ["score", "selectivity_gap", "hotspot_overlap"], ascending=False).head(20)
    top_per.append(g2)
top = pd.concat(top_per, ignore_index=True)
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top<=20 per paradigm)")
print(top.groupby("paradigm").size().to_dict())
top.head(8)[["design_id", "paradigm", "score", "pae_interaction", "rosetta_dG",
             "selectivity_gap", "hotspot_overlap"]]

wrote results/top_candidates.csv: (8, 23) (top<=20 per paradigm)
{'bindcraft': 1, 'rfdiffusion': 7}


,design_id,paradigm,score,pae_interaction,rosetta_dG,selectivity_gap,hotspot_overlap
0,EXAMPLE_DATA_bindcraft_0028,bindcraft,3.2133,5.0,-34.0,-2.0,0.2
1,EXAMPLE_DATA_rfdiffusion_0079,rfdiffusion,4.5133,5.0,-44.0,3.0,1.0
2,EXAMPLE_DATA_rfdiffusion_0167,rfdiffusion,3.8033,8.0,-36.0,0.0,1.0
3,EXAMPLE_DATA_rfdiffusion_0007,rfdiffusion,3.7667,7.0,-45.0,-2.0,0.2
4,EXAMPLE_DATA_rfdiffusion_0161,rfdiffusion,3.7000,9.0,-35.0,-10.0,0.4
5,EXAMPLE_DATA_rfdiffusion_0042,rfdiffusion,3.1700,6.0,-36.0,-2.0,0.2
6,EXAMPLE_DATA_rfdiffusion_0051,rfdiffusion,3.1167,10.0,-30.0,-6.0,1.0
7,EXAMPLE_DATA_rfdiffusion_0131,rfdiffusion,2.6967,10.0,-39.0,-3.0,0.6


## D3 (part 2) checklist
- [ ] Head-to-head: hit rate + interface-energy distribution per paradigm (figure `results/p08_headtohead.png`).
- [ ] **Isoform-specificity panel: selectivity gap per survivor + per paradigm** (`results/isoform_specificity.csv`, `results/p08_selectivity.png`).
- [ ] Novelty compared across paradigms (TM-score to PDB) — or the scaffold wired up on Colab.
- [ ] Effector-competition vs RAF: switch-footprint coverage of survivors; "likely blocker" count.
- [ ] `results/top_candidates.csv`: top 10–20 each, preferring selective blockers, ready for the validation plan.
- [ ] Honest discussion: the two paradigms' failure modes **and** why isoform selectivity is hard (not just a winner).

**Next:** `05_validation_plan.ipynb` — the SPR/BLI + isoform-specificity + nucleotide-state plan.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — SPR/BLI + isoform-specificity panel + nucleotide-state test + controls

**Standard slot:** *validation plan.* **For Project 08 this means:** turn the top candidates into a
**costed, controlled wet-lab plan** — SPR/BLI affinity vs KRAS, an **isoform-specificity panel**
(KRAS/HRAS/NRAS), a **nucleotide-state-dependence test** (GDP- vs GppNHp-loaded KRAS) `[stretch]`, the
mandatory controls (positive known KRAS binder, **scrambled-interface** negative, unrelated negative),
an expression strategy (nucleotide-loaded reagent), and the **Boltz-2 affinity** stretch (scaffold only)
(D4/D5).

A design that passes every filter is a **hypothesis** — SPR/BLI + the isoform panel are what test it.
Needs `results/top_candidates.csv` (notebook 04).

## Setup paths

In [33]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_08_kras_binder/notebooks


## 1 · Draft the experimental validation plan

Generate a plan card from the top candidates: assays, the isoform panel, the nucleotide-state test,
controls, expression, timeline, costed reagents. Fill the `<...>` from your own numbers; this is the
deliverable other people will actually read.

In [34]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_par = top.groupby("paradigm").size().to_dict() if n_top else {}

plan = f"""# KRAS Binder Validation Plan (Project 08 — by <your name>, <date>)

## Candidates
Top {n_top} candidates carried forward ({by_par}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured — `pae_interaction` is confidence (not affinity),
and the in-silico selectivity gap is NOT measured selectivity. No K_D is reported here.

## Expression strategy
- Binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (50-90 aa) -> high yield expected.
- KRAS reagent: express the G-domain (res ~1-169); NUCLEOTIDE-LOAD it deliberately -> prepare BOTH
  GDP-loaded and GppNHp/GMPPCP-loaded KRAS (keep Mg2+). Confirm each is folded/active before testing binders.
- For the isoform panel: express/obtain HRAS and NRAS G-domains under the SAME conditions.

## Assays (go/no-go -> basic -> selectivity/functional)
1. Go/no-go: express -> SDS-PAGE -> SEC (monodisperse?).
2. Affinity: SPR or BLI vs immobilized KRAS -> K_D + kinetics (k_on/k_off). Test a dilution series.
3. ISOFORM-SPECIFICITY PANEL (the centerpiece): run the SAME binder vs KRAS, HRAS, and NRAS under
   identical conditions -> quantify selectivity. A pan-RAS binder is a weaker result; report it.
4. NUCLEOTIDE-STATE TEST [stretch]: compare binding to GDP-loaded vs GppNHp-loaded KRAS -> a
   state-specific binder should discriminate.
5. Functional (extension): effector competition -> does the binder block RAF-RBD binding to KRAS-GTP?
6. Stability: DSF (Tm). Deep (optional): co-crystal / cryo-EM; cell-based KRAS-pathway readout.

## Controls (MANDATORY)
- Positive: a known KRAS binder (published DARPin/monobody/binder, or a G12C-inhibitor complex as a
  state reference) -> assay + KRAS reagent are active.
- Negative (scrambled-interface): YOUR OWN top design with its interface residues scrambled/mutated
  -> must LOSE binding (cleanest specificity control).
- Negative (unrelated): an unrelated mini-protein of similar size -> should not bind.
- Isoform off-targets (HRAS/NRAS) double as the selectivity readout AND a specificity control.

## Realistic expectations
KRAS is a hard target; SELECTIVITY (isoform + allele) is harder still. In-silico hit rates vary widely
and the MAJORITY of in-silico hits fail experimentally. Expect to test many to find a few real, and
fewer still selective, binders. Report the experimental hit rate AND the measured selectivity honestly.
Do NOT imply a working/selective binder or fabricate a K_D.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} binders + scrambled-interface negatives): $<...>, <...> weeks (IGSC-screened provider).
- KRAS/HRAS/NRAS reagents + nucleotide loading (GDP, GppNHp) + SPR/BLI chips + positive control: $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
Inhibitory/blocking binders to KRAS, a human oncotarget, for cancer therapeutics/diagnostics (in scope).
Gene synthesis via a biosecurity-screening provider; wet lab under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:600], "...")

wrote results/validation_plan.md — fill the <...> placeholders from your numbers.
# KRAS Binder Validation Plan (Project 08 — by <your name>, <date>)

## Candidates
Top 8 candidates carried forward ({'bindcraft': 1, 'rfdiffusion': 7}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured — `pae_interaction` is confidence (not affinity),
and the in-silico selectivity gap is NOT measured selectivity. No K_D is reported here.

## Expression strategy
- Binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (50-90 aa) -> high yield expected.
- KRAS reagent: express the G-domain (res ~1-169); NUCLEOTIDE-LOAD it deliberately - ...


## 2 · Build the scrambled-interface negative controls

The single cleanest specificity control: take each top design and **scramble its interface residues**
(the positions contacting KRAS) — it should **lose** binding. Generating these alongside the real designs
(same expression batch) makes the SPR/BLI comparison airtight. Here we scaffold the sequence-level
scramble deterministically; on Colab, scramble the *interface* positions specifically using the predicted
contacts.

In [35]:
import random
import binder_tools as bt   # bt._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

def scramble_interface(seq, frac=0.4, seed=0):
    """Deterministically shuffle a fraction of the sequence as a NEGATIVE-CONTROL stand-in.
    On Colab, scramble the predicted INTERFACE residues specifically (positions contacting KRAS)."""
    rng = random.Random(seed)
    seq = list(seq)
    idx = list(range(len(seq)))
    rng.shuffle(idx)
    k = max(1, int(len(seq) * frac))
    chosen = idx[:k]
    vals = [seq[i] for i in chosen]
    rng.shuffle(vals)
    for i, v in zip(chosen, vals):
        seq[i] = v
    return "".join(seq)

negs = []
if n_top and "sequence" in top.columns:
    for _, r in top.iterrows():
        s = str(r.get("sequence", ""))
        if s and s != "nan":
            negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                             parent=r["design_id"], paradigm=r.get("paradigm"),
                             sequence=scramble_interface(s, seed=bt._hashints(r["design_id"]) % 10**6),
                             role="scrambled-interface negative control"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-interface negatives")
else:
    print("Run notebook 04 first to produce results/top_candidates.csv with sequences.")

wrote results/negative_controls.csv: 8 scrambled-interface negatives


## 3 · (Stretch) Boltz-2 affinity on top hits `[stretch]`

Boltz-2 can predict a binding-affinity signal for the top complexes. Use it for **relative ranking +
caveats only** — **never fabricate a K_D**, and never present a predicted number as measured. This tells
you which hits to test first, not whether they bind (or are selective).

In [36]:
# Scaffold ONLY. Do NOT invent affinities. On Colab:
#   pip install boltz; build the (binder, KRAS) complex input; run boltz predict with affinity mode;
#   read the predicted-affinity signal and report the RELATIVE ranking of the top hits + heavy caveats.
#   You can also run it per isoform to prioritize the most KRAS-selective hits (still relative-only).
# Pinned upstream (verify): https://github.com/jwohlwend/boltz
print("Boltz-2 affinity is a STRETCH scaffold: relative ranking + caveats only, NEVER a fabricated K_D.")
print("Use it to PRIORITIZE which top hits to test first in SPR/BLI + the isoform panel — not as evidence of binding.")

Boltz-2 affinity is a STRETCH scaffold: relative ranking + caveats only, NEVER a fabricated K_D.
Use it to PRIORITIZE which top hits to test first in SPR/BLI + the isoform panel — not as evidence of binding.


## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: SPR/BLI + **isoform-specificity panel** + **nucleotide-state test**, expression (nucleotide-loaded KRAS), timeline, costed reagents.
- [ ] Controls specified: positive (known KRAS binder), **scrambled-interface** negative (`results/negative_controls.csv`), unrelated negative; HRAS/NRAS off-targets double as the selectivity readout.
- [ ] (Stretch) Boltz-2 affinity used only for relative ranking, with caveats — no fabricated K_D.
- [ ] Honest framing: every design is a hypothesis until SPR/BLI + the isoform panel; report the experimental hit rate AND measured selectivity.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a rigorous, honestly-reported KRAS binder campaign whose centerpiece is **selectivity**.